In [ ]:
!pip install -U "chromadb<0.6.0" "opentelemetry-sdk<1.27.0" "opentelemetry-api<1.27.0"
!pip install -U langchain_chroma

  Using cached chromadb-0.5.23-py3-none-any.whl.metadata (6.8 kB)
Using cached chromadb-0.5.23-py3-none-any.whl (628 kB)
  Attempting uninstall: chromadb
    Found existing installation: chromadb 1.5.9
    Uninstalling chromadb-1.5.9:
      Successfully uninstalled chromadb-1.5.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires chromadb<2.0.0,>=1.3.5, but you have chromadb 0.5.23 which is incompatible.
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (23.3 MB)
  Attempting uninstall: chromadb
    Found existing installation: chromadb 0.5.23
    Uninstalling chromadb-0.5.23:
      Successfully uninstalled chromadb-0.5.23


In [ ]:
!pip install langchain_google_genai
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from google.colab import userdata
from langchain_chroma import Chroma
from pydantic import BaseModel, Field

#STEP2 - Load api key
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

#STEP 3 - Initialize LLM and Embeddings Model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Correctly initialize the embeddings model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

class EmailClassificationOutput(BaseModel):
    urgency: str = Field(description="Classification of email urgency: High, Medium, or Low.")
    topic: str = Field(description="Main topic of discussion in the email (e.g., Accounts, Billing, Password reset).")
    response_text_category: str = Field(description="Category of response needed (e.g., Simple, Complex/Unresolved, needs customer support analyst).")
    follow_up_required: bool = Field(description="True if a follow-up action is required, False otherwise.")

class EscalationEmailOutput(BaseModel):
    recipient: str = Field(description="The email address of the internal customer support agent or team.")
    subject: str = Field(description="The subject line for the escalation email.")
    body: str = Field(description="The body of the escalation email, detailing the issue and reason for escalation.")

In [ ]:
class AnalysisOutput(BaseModel):
    analysis: str = Field(description="Detailed analysis of the email content and retrieved documents.")
    response_draft: str = Field(description="A draft response to the email based on the analysis and retrieved information.")
    follow_up_required: bool = Field(description="True if additional follow-up action is required after sending the drafted response, False otherwise.")

In [ ]:
from typing import TypedDict, Optional, List
from langchain_core.documents import Document

# Define the AgentState for managing the agent's workflow
class AgentState(TypedDict):
    email_content: str # Input email content for classification
    urgency: Optional[EmailClassificationOutput] # Using the previously defined structured output
    analysis: Optional[str]
    response_draft: Optional[str]
    follow_up_required: Optional[bool]
    retrieved_docs: Optional[List[Document]] # New field to store retrieved documents
    escalation_required: Optional[bool] # New field to indicate if escalation is needed
    escalation_email_draft: Optional[str] # New field for the drafted escalation email

In [ ]:
#!pip install -U langchain langchain-community langchain-text-splitters
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
kb_path = '/content/knowledge_base.rtf'
loader = TextLoader(kb_path)
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
vector_db = Chroma.from_documents(splitter.split_documents(docs), embeddings)
retriever = vector_db.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# Define a system prompt for urgency identification
classify_system_prompt = """
You are an AI assistant tasked with analyzing email content and extracting structured information.
Your output MUST be a JSON object that adheres to the following schema:

```json
{schema}
```

Based on the email content, you need to classify the following:
1.  **Urgency**: Classify its urgency as one of the following categories:
    -   High: Requires immediate attention, critical, time-sensitive, potential for significant negative impact if delayed.
    -   Medium: Important, needs attention within a day or two, but not immediately critical.
    -   Low: Informational, non-urgent, can be addressed at convenience.

2.  **Topic**: Identify the main topic of discussion in the email (e.g., Accounts, Accounts, Billing, Password reset, Product Inquiry, Technical Issue, etc.).

3.  **Response Text Category**: Determine the category of response needed (e.g., Simple, Complex/Unresolved, needs customer support analyst, FAQ-answerable, etc.).

4.  **Follow-up Required**: Indicate whether a follow-up action is required (True/False).
"""

# Create a ChatPromptTemplate for urgency classification
classify_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", classify_system_prompt),
        ("user", "{email_content}")
    ]
)

# Define a sample email for demonstration
sample_email_high_urgency = "Subject: CRITICAL: Server Down in Production!\n\nTeam, the main production server went down 5 minutes ago. All services are affected. This is impacting our core business. Please respond immediately!"
sample_email_medium_urgency = "Subject: Follow-up on Q3 Report\n\nHi team, just a reminder to finalize your contributions for the Q3 report by end of day tomorrow. Please let me know if you have any blockers."
sample_email_low_urgency = "Subject: Newsletter: Upcoming Company Picnic\n\nHello all, don't forget about the annual company picnic next month! More details to follow soon."

# Create a chain with structured output
urgency_chain = classify_prompt_template | llm.with_structured_output(EmailClassificationOutput)

escalation_email_system_prompt = """
You are an AI assistant tasked with drafting an internal escalation email to a customer support agent.
Your output MUST be a JSON object that adheres to the following schema:

```json
{schema}
```

Based on the provided email content, its urgency classification, and any analysis:
1.  **Recipient**: Specify the email address of the internal customer support team or agent (e.g., 'support@example.com', 'noc@example.com').
2.  **Subject**: Create a clear and concise subject line for the escalation email, including the original email's subject and its urgency.
3.  **Body**: Draft a detailed body for the escalation email. This should include:
    -   A summary of the original email's problem.
    -   The determined urgency and topic.
    -   Any relevant analysis or context that might help the support agent.
    -   A clear request for the support agent to take over or provide guidance.

Original Email Content: {email_content}
Urgency Classification: {urgency_classification}
Analysis (if available): {analysis}
"""

escalation_email_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", escalation_email_system_prompt),
        ("user", "Draft an escalation email for this issue.")
    ]
)

escalation_email_chain = escalation_email_prompt_template | llm.with_structured_output(EscalationEmailOutput)

In [ ]:
analysis_system_prompt = """
You are an AI assistant tasked with analyzing an email and relevant retrieved documents to formulate a comprehensive analysis, draft a response, and determine if further follow-up is necessary.
Your output MUST be a JSON object that adheres to the following schema:

```json
{schema}
```

Based on the user's email and the provided context:
1.  **Analysis**: Provide a detailed analysis of the email's core issue, incorporating information from the retrieved documents.
2.  **Response Draft**: Generate a concise and helpful draft response to the user.
3.  **Follow-up Required**: Indicate whether any further action or follow-up is needed after this response (True/False).

Email Content: {email_content}

Retrieved Documents: {retrieved_docs}
"""

analysis_draft_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", analysis_system_prompt),
        ("user", "Analyze the email and context to draft a response.")
    ]
)

analysis_draft_chain = analysis_draft_prompt_template | llm.with_structured_output(AnalysisOutput)

In [ ]:
import json # Import the json module

def escalation_node(state: AgentState) -> dict:
    """Determines if an email requires escalation based on its urgency classification."""
    urgency_classification = state.get("urgency")
    escalation = False

    if urgency_classification:
        # Escalation if urgency is High or if response category indicates complex/support needed
        if urgency_classification.urgency == "High" or \
           urgency_classification.response_text_category in ["Complex/Unresolved", "needs customer support analyst"]:
            escalation = True

    return {"escalation_required": escalation}

def draft_escalation_email_node(state: AgentState) -> dict:
    """Drafts an internal escalation email based on the current state."""
    email_content = state["email_content"]
    urgency_classification = state.get("urgency")
    analysis = state.get("analysis")

    escalation_output: EscalationEmailOutput = escalation_email_chain.invoke({
        "email_content": email_content,
        "urgency_classification": urgency_classification.model_dump_json() if urgency_classification else "N/A", # Pass as JSON string
        "analysis": analysis if analysis else "No specific analysis available.",
        "schema": json.dumps(EscalationEmailOutput.model_json_schema()) # Updated to use model_json_schema() and json.dumps()
    })

    return {"escalation_email_draft": f"To: {escalation_output.recipient}\nSubject: {escalation_output.subject}\n\n{escalation_output.body}"}

# --- Demonstration of escalation_node ---
print("--- Demonstrating escalation_node ---")

# Example 1: High urgency email (should require escalation)
mock_high_urgency_output = EmailClassificationOutput(
    urgency="High",
    topic="Server Outage",
    response_text_category="needs customer support analyst",
    follow_up_required=True
)
state_high_urgency = AgentState(
    email_content=sample_email_high_urgency,
    urgency=mock_high_urgency_output,
    analysis=None, response_draft=None, follow_up_required=None, retrieved_docs=None, escalation_required=None
)
escalation_updates_high = escalation_node(state_high_urgency)
print(f"High Urgency Email Escalation Required: {escalation_updates_high['escalation_required']}")

# Example 2: Medium urgency email (might not require escalation if category is simple)
mock_medium_urgency_output = EmailClassificationOutput(
    urgency="Medium",
    topic="Report Query",
    response_text_category="Simple",
    follow_up_required=True
)
state_medium_urgency = AgentState(
    email_content=sample_email_medium_urgency,
    urgency=mock_medium_urgency_output,
    analysis=None, response_draft=None, follow_up_required=None, retrieved_docs=None, escalation_required=None
)
escalation_updates_medium = escalation_node(state_medium_urgency)
print(f"Medium Urgency Email Escalation Required: {escalation_updates_medium['escalation_required']}")

# Example 3: Low urgency email (should not require escalation)
mock_low_urgency_output = EmailClassificationOutput(
    urgency="Low",
    topic="Company Event",
    response_text_category="Informational",
    follow_up_required=False
)
state_low_urgency = AgentState(
    email_content=sample_email_low_urgency,
    urgency=mock_low_urgency_output,
    analysis=None, response_draft=None, follow_up_required=None, retrieved_docs=None, escalation_required=None
)
escalation_updates_low = escalation_node(state_low_urgency)
print(f"Low Urgency Email Escalation Required: {escalation_updates_low['escalation_required']}")

# --- Demonstration of draft_escalation_email_node ---
print("\n--- Demonstrating draft_escalation_email_node ---")

# Use a state that indicates escalation is required
state_for_escalation_draft = AgentState(
    email_content=sample_email_high_urgency,
    urgency=mock_high_urgency_output,
    analysis="The server is down affecting all services. Immediate NOC intervention is needed.",
    response_draft=None, follow_up_required=True, retrieved_docs=None, escalation_required=True, escalation_email_draft=None
)
escalation_draft_result = draft_escalation_email_node(state_for_escalation_draft)
print(f"Escalation Email Draft:\n{escalation_draft_result['escalation_email_draft']}")

--- Demonstrating escalation_node ---
High Urgency Email Escalation Required: True
Medium Urgency Email Escalation Required: False
Low Urgency Email Escalation Required: False

--- Demonstrating draft_escalation_email_node ---
Escalation Email Draft:
To: noc@example.com
Subject: Escalation: High Urgency - CRITICAL: Server Down in Production!

Hello Team,

This is an urgent escalation regarding a critical server outage. The main production server went down 5 minutes ago, affecting all services and impacting our core business operations.

Urgency: High
Topic: Server Outage

Analysis indicates that immediate NOC intervention is critically needed to restore services. Please take over this high-priority incident immediately and provide an update on the resolution plan.


In [ ]:
def follow_up_node(state: AgentState) -> dict:
    """Determines if a follow-up is required based on the current state."""
    # The follow_up_required flag is already set by the analyze_and_draft_node
    # This node primarily acts as a checkpoint or a way to make the decision explicit
    is_follow_up_required = state.get("follow_up_required", False)
    return {"follow_up_required": is_follow_up_required}

# --- Demonstration of follow_up_node ---
print("--- Demonstrating follow_up_node ---")

# Example 1: With high urgency email, where analysis indicated follow-up
mock_analysis_output_high = AnalysisOutput(
    analysis="High urgency email, needs immediate attention.",
    response_draft="Drafted urgent response.",
    follow_up_required=True
)
state_with_analysis_high = AgentState(
    email_content="mock content",
    urgency=None, analysis=mock_analysis_output_high.analysis,
    response_draft=mock_analysis_output_high.response_draft,
    follow_up_required=mock_analysis_output_high.follow_up_required,
    retrieved_docs=None, escalation_required=None
)
follow_up_updates_high = follow_up_node(state_with_analysis_high)
print(f"High Urgency Email Follow-up Required: {follow_up_updates_high['follow_up_required']}")

# Example 2: With low urgency email, where analysis indicated no follow-up
mock_analysis_output_low = AnalysisOutput(
    analysis="Low urgency email, no immediate action needed.",
    response_draft="Drafted casual response.",
    follow_up_required=False
)
state_with_analysis_low = AgentState(
    email_content="mock content",
    urgency=None, analysis=mock_analysis_output_low.analysis,
    response_draft=mock_analysis_output_low.response_draft,
    follow_up_required=mock_analysis_output_low.follow_up_required,
    retrieved_docs=None, escalation_required=None
)
follow_up_updates_low = follow_up_node(state_with_analysis_low)
print(f"Low Urgency Email Follow-up Required: {follow_up_updates_low['follow_up_required']}")

--- Demonstrating follow_up_node ---
High Urgency Email Follow-up Required: True
Low Urgency Email Follow-up Required: False


In [ ]:
import json # Import the json module
from langgraph.graph import StateGraph, END
from typing import Optional, List
from langchain_core.documents import Document

# Definition for classify_email_urgency_node (from cell c4ca8651)
def classify_email_urgency_node(state: AgentState) -> dict:
  """Classifies the urgency of an email based on its content, returning a dictionary of state updates."""
  email_content = state["email_content"]
  # Updated to use model_json_schema() and json.dumps()
  response_obj: EmailClassificationOutput = urgency_chain.invoke({"email_content": email_content, "schema": json.dumps(EmailClassificationOutput.model_json_schema())})

  # Return a dictionary to update the AgentState
  return {
      "urgency": response_obj,
      "follow_up_required": response_obj.follow_up_required
  }

# Definition for rag_node (from cell q7FLv1lWUZPU)
def rag_node(state: AgentState) -> dict:
  """Retrieves relevant documents from the vector database based on email content.
  Assumes 'retriever' is initialized globally or passed into the function.
  """
  email_content = state["email_content"]
  docs = retriever.invoke(email_content)
  return {"retrieved_docs": docs}

# Definition for analyze_and_draft_node (from cell 5b58077f)
def analyze_and_draft_node(state: AgentState) -> dict:
    """Analyzes the email and retrieved documents to draft a response and determine follow-up."""
    email_content = state["email_content"]
    retrieved_docs = state["retrieved_docs"]

    # Invoke the analysis chain
    # Updated to use model_json_schema() and json.dumps()
    analysis_output: AnalysisOutput = analysis_draft_chain.invoke({
        "email_content": email_content,
        "retrieved_docs": retrieved_docs,
        "schema": json.dumps(AnalysisOutput.model_json_schema())
    })

    # Update the AgentState with the analysis, response draft, and follow-up status
    return {
        "analysis": analysis_output.analysis,
        "response_draft": analysis_output.response_draft,
        "follow_up_required": analysis_output.follow_up_required
    }

# Define the graph
workflow = StateGraph(AgentState)

# Add the nodes to the graph
workflow.add_node("classify_urgency", classify_email_urgency_node)
workflow.add_node("retrieve_docs", rag_node)
workflow.add_node("escalate_check", escalation_node)
workflow.add_node("draft_escalation_email", draft_escalation_email_node)
workflow.add_node("analyze_and_draft", analyze_and_draft_node)
workflow.add_node("follow_up_check", follow_up_node)

# Set the entry point
workflow.set_entry_point("classify_urgency")

# Add edges to define the sequential flow
workflow.add_edge("classify_urgency", "retrieve_docs")
workflow.add_edge("retrieve_docs", "escalate_check")

# Conditional edge after escalate_check
workflow.add_conditional_edges(
    "escalate_check",
    lambda state: "draft_escalation_email" if state.get("escalation_required") else "analyze_and_draft",
    {
        "draft_escalation_email": "draft_escalation_email",
        "analyze_and_draft": "analyze_and_draft"
    }
)

# Connect the escalation path and the normal path to the follow_up_check
workflow.add_edge("draft_escalation_email", "follow_up_check")
workflow.add_edge("analyze_and_draft", "follow_up_check")

# The final node in this sequence leads to the END state
workflow.add_edge("follow_up_check", END)

# Compile the graph
app = workflow.compile()

print("Graph compiled successfully!")

# --- Demonstration of the compiled graph ---
print("\n--- Demonstrating the full workflow with a high urgency email ---")

# Initial state for a high urgency email
initial_email_high = sample_email_high_urgency
initial_state_full_high = AgentState(
    email_content=initial_email_high,
    urgency=None, analysis=None, response_draft=None, follow_up_required=None,
    retrieved_docs=None, escalation_required=None, escalation_email_draft=None
)

# Run the graph
final_state_high = app.invoke(initial_state_full_high)

print("\n--- Final State for High Urgency Email ---")
print(f"Urgency: {final_state_high.get('urgency').urgency if final_state_high.get('urgency') else 'N/A'}")
print(f"Topic: {final_state_high.get('urgency').topic if final_state_high.get('urgency') else 'N/A'}")
print(f"Response Category: {final_state_high.get('urgency').response_text_category if final_state_high.get('urgency') else 'N/A'}")
print(f"Retrieved Docs Count: {len(final_state_high.get('retrieved_docs')) if final_state_high.get('retrieved_docs') else 0}")
print(f"Escalation Required: {final_state_high.get('escalation_required')}")
print(f"Escalation Email Draft: {final_state_high.get('escalation_email_draft')}")
print(f"Analysis: {final_state_high.get('analysis')}")
print(f"Response Draft: {final_state_high.get('response_draft')}")
print(f"Follow-up Required: {final_state_high.get('follow_up_required')}")

print("\n--- Demonstrating the full workflow with a low urgency email ---")

# Initial state for a low urgency email
initial_email_low = sample_email_low_urgency
initial_state_full_low = AgentState(
    email_content=initial_email_low,
    urgency=None, analysis=None, response_draft=None, follow_up_required=None,
    retrieved_docs=None, escalation_required=None, escalation_email_draft=None
)

# Run the graph
final_state_low = app.invoke(initial_state_full_low)

print("\n--- Final State for Low Urgency Email ---")
print(f"Urgency: {final_state_low.get('urgency').urgency if final_state_low.get('urgency') else 'N/A'}")
print(f"Topic: {final_state_low.get('urgency').topic if final_state_low.get('urgency') else 'N/A'}")
print(f"Response Category: {final_state_low.get('urgency').response_text_category if final_state_low.get('urgency') else 'N/A'}")
print(f"Retrieved Docs Count: {len(final_state_low.get('retrieved_docs')) if final_state_low.get('retrieved_docs') else 0}")
print(f"Escalation Required: {final_state_low.get('escalation_required')}")
print(f"Escalation Email Draft: {final_state_low.get('escalation_email_draft')}")
print(f"Analysis: {final_state_low.get('analysis')}")
print(f"Response Draft: {final_state_low.get('response_draft')}")
print(f"Follow-up Required: {final_state_low.get('follow_up_required')}")

Graph compiled successfully!

--- Demonstrating the full workflow with a high urgency email ---

--- Final State for High Urgency Email ---
Urgency: High
Topic: Technical Issue
Response Category: Complex/Unresolved
Retrieved Docs Count: 4
Escalation Required: True
Escalation Email Draft: To: noc@example.com
Subject: Escalation: HIGH URGENCY - CRITICAL: Server Down in Production!

Hello Team,This is an urgent escalation regarding a critical production issue. The main production server went down approximately 5 minutes ago. All services are currently affected, impacting our core business operations.The urgency for this issue is classified as 'High', and the topic is 'Technical Issue'. No specific analysis is available at this time beyond the initial report.Please take immediate action to investigate and resolve this server outage. We require an immediate response and resolution plan.
Analysis: None
Response Draft: None
Follow-up Required: True

--- Demonstrating the full workflow with a 

In [ ]:
def classify_email_urgency_node(state: AgentState) -> dict:
  """Classifies the urgency of an email based on its content, returning a dictionary of state updates."""
  email_content = state["email_content"]
  response_obj: EmailClassificationOutput = urgency_chain.invoke({"email_content": email_content, "schema": EmailClassificationOutput.schema_json()})

  # Return a dictionary to update the AgentState
  return {
      "urgency": response_obj, # Store the whole EmailClassificationOutput object
      "follow_up_required": response_obj.follow_up_required
  }

# Demonstrate the use of the new node function
print("--- Using the classify_email_urgency_node function ---")

# High urgency email demonstration
initial_state_high = AgentState(email_content=sample_email_high_urgency, urgency=None, analysis=None, response_draft=None, follow_up_required=None)
classified_updates_high = classify_email_urgency_node(initial_state_high)
# Simulate state update for demonstration
high_urgency_classified = classified_updates_high["urgency"]

print(f"High Urgency Email Classification: {high_urgency_classified.urgency}")
print(f"High Urgency Email Topic: {high_urgency_classified.topic}")
print(f"High Urgency Email Response Category: {high_urgency_classified.response_text_category}")
print(f"High Urgency Email Follow-up Required: {high_urgency_classified.follow_up_required}")

# Medium urgency email demonstration
initial_state_medium = AgentState(email_content=sample_email_medium_urgency, urgency=None, analysis=None, response_draft=None, follow_up_required=None)
classified_updates_medium = classify_email_urgency_node(initial_state_medium)
medium_urgency_classified = classified_updates_medium["urgency"]

print(f"\nMedium Urgency Email Classification: {medium_urgency_classified.urgency}")
print(f"Medium Urgency Email Topic: {medium_urgency_classified.topic}")
print(f"Medium Urgency Email Response Category: {medium_urgency_classified.response_text_category}")
print(f"Medium Urgency Email Follow-up Required: {medium_urgency_classified.follow_up_required}")

# Low urgency email demonstration
initial_state_low = AgentState(email_content=sample_email_low_urgency, urgency=None, analysis=None, response_draft=None, follow_up_required=None)
classified_updates_low = classify_email_urgency_node(initial_state_low)
low_urgency_classified = classified_updates_low["urgency"]

print(f"\nLow Urgency Email Classification: {low_urgency_classified.urgency}")
print(f"Low Urgency Email Topic: {low_urgency_classified.topic}")
print(f"Low Urgency Email Response Category: {low_urgency_classified.response_text_category}")
print(f"Low Urgency Email Follow-up Required: {low_urgency_classified.follow_up_required}")

--- Using the classify_email_urgency_node function ---


/tmp/ipykernel_4035/1724508182.py:4: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  response_obj: EmailClassificationOutput = urgency_chain.invoke({"email_content": email_content, "schema": EmailClassificationOutput.schema_json()})


High Urgency Email Classification: High
High Urgency Email Topic: Technical Issue
High Urgency Email Response Category: needs customer support analyst
High Urgency Email Follow-up Required: True

Medium Urgency Email Classification: Medium
Medium Urgency Email Topic: Reporting
Medium Urgency Email Response Category: Simple
Medium Urgency Email Follow-up Required: True

Low Urgency Email Classification: Low
Low Urgency Email Topic: Company Event
Low Urgency Email Response Category: Informational
Low Urgency Email Follow-up Required: False


In [ ]:
def rag_node(state: AgentState) -> dict:
  """Retrieves relevant documents from the vector database based on email content.
  Assumes 'retriever' is initialized globally or passed into the function.
  """
  email_content = state["email_content"]
  docs = retriever.invoke(email_content)
  return {"retrieved_docs": docs}

# Demonstrate the use of the rag_node
print("--- Demonstrating rag_node ---")

# Example with high urgency email
initial_state_for_rag = AgentState(email_content=sample_email_high_urgency, urgency=None, analysis=None, response_draft=None, follow_up_required=None, retrieved_docs=None)
rag_updates = rag_node(initial_state_for_rag)

print(f"Retrieved documents for high urgency email: {rag_updates['retrieved_docs']}")

# Example with low urgency email
initial_state_for_rag_low = AgentState(email_content=sample_email_low_urgency, urgency=None, analysis=None, response_draft=None, follow_up_required=None, retrieved_docs=None)
rag_updates_low = rag_node(initial_state_for_rag_low)

print(f"\nRetrieved documents for low urgency email: {rag_updates_low['retrieved_docs']}")

--- Demonstrating rag_node ---
Retrieved documents for high urgency email: [Document(id='5c11ea09-a36c-431e-97b7-d4ffc7b96919', metadata={'source': '/content/knowledge_base.rtf'}, page_content='\\\n## Technical Issues - General\\\n- Clear your browser cache and cookies before reporting issues.\\\n- Ensure you are using a supported browser: Chrome 90+, Firefox 88+, Safari 14+, Edge 90+.\\\n- Disable browser extensions that may interfere with the application.\\\n- Check our status page at https://status.company.com for ongoing incidents.\\\n- For mobile app issues, ensure you have the latest version installed from the App Store or Google Play.}'), Document(id='12ed63c0-43eb-4dac-a07c-1e06fd3cb8e6', metadata={'source': '/content/knowledge_base.rtf'}, page_content='\\\n## Dark Mode\\\n- Dark mode is currently available on the web application (Settings > Appearance > Dark Mode).\\\n- Dark mode for the mobile app (iOS and Android) is on our product roadmap and planned for Q3 2025.\\\n- You c

In [ ]:
def analyze_and_draft_node(state: AgentState) -> dict:
    """Analyzes the email and retrieved documents to draft a response and determine follow-up."""
    email_content = state["email_content"]
    retrieved_docs = state["retrieved_docs"]

    # Invoke the analysis chain
    analysis_output: AnalysisOutput = analysis_draft_chain.invoke({
        "email_content": email_content,
        "retrieved_docs": retrieved_docs,
        "schema": AnalysisOutput.schema_json()
    })

    # Update the AgentState with the analysis, response draft, and follow-up status
    return {
        "analysis": analysis_output.analysis,
        "response_draft": analysis_output.response_draft,
        "follow_up_required": analysis_output.follow_up_required
    }

# --- Demonstration of analyze_and_draft_node ---
print("--- Demonstrating analyze_and_draft_node ---")

# Create a mock state for demonstration (assuming rag_node has already run)
mock_retrieved_docs_high = [
    Document(page_content="Server maintenance schedule: Production servers are critical and require immediate attention for any outages."),
    Document(page_content="Incident response protocol: For P1 incidents (like server outages), escalate immediately to the NOC team and inform stakeholders.")
]
mock_retrieved_docs_low = [
    Document(page_content="Company picnic details: Annual event, usually held in June. Details are sent out via internal newsletter.")
]

# High urgency email with mock retrieved docs
state_with_rag_high = AgentState(
    email_content=sample_email_high_urgency,
    urgency=None, analysis=None, response_draft=None, follow_up_required=None,
    retrieved_docs=mock_retrieved_docs_high
)
analysis_updates_high = analyze_and_draft_node(state_with_rag_high)

print("\n--- High Urgency Email Analysis ---")
print(f"Analysis: {analysis_updates_high['analysis']}")
print(f"Response Draft: {analysis_updates_high['response_draft']}")
print(f"Follow-up Required: {analysis_updates_high['follow_up_required']}")

# Low urgency email with mock retrieved docs
state_with_rag_low = AgentState(
    email_content=sample_email_low_urgency,
    urgency=None, analysis=None, response_draft=None, follow_up_required=None,
    retrieved_docs=mock_retrieved_docs_low
)
analysis_updates_low = analyze_and_draft_node(state_with_rag_low)

print("\n--- Low Urgency Email Analysis ---")
print(f"Analysis: {analysis_updates_low['analysis']}")
print(f"Response Draft: {analysis_updates_low['response_draft']}")
print(f"Follow-up Required: {analysis_updates_low['follow_up_required']}")

--- Demonstrating analyze_and_draft_node ---


/tmp/ipykernel_4035/2597000354.py:10: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  "schema": AnalysisOutput.schema_json()



--- High Urgency Email Analysis ---
Analysis: The email reports a critical P1 incident: the main production server is down, affecting all services and core business operations. This aligns with the retrieved document stating 'Production servers are critical and require immediate attention for any outages'. The incident response protocol specifies that 'For P1 incidents (like server outages), escalate immediately to the NOC team and inform stakeholders.' Therefore, the immediate action required is to engage the NOC team and begin stakeholder communication.
Response Draft: Subject: RE: CRITICAL: Server Down in Production!Team,Thank you for the immediate alert. We understand the critical nature of this outage. The Network Operations Center (NOC) team has been immediately engaged and is actively investigating the production server issue as per our P1 incident protocol. We will provide updates as soon as more information is available and will ensure all relevant stakeholders are kept infor